# Evaluate Simba PPO Model

This notebook evaluates a trained Simba PPO model on the Flycraft environment.

In [12]:
import sys
from pathlib import Path
import argparse

# Add project root to path
PROJECT_ROOT_DIR = Path().absolute().parent
if str(PROJECT_ROOT_DIR.absolute()) not in sys.path:
    sys.path.append(str(PROJECT_ROOT_DIR.absolute()))

from utils_my.models.ppo_with_bc_loss import PPOWithBCLoss
from utils_my.models.simba_policy import SimBaActorCriticPolicy
from utils_my.sb3.vec_env_helper import get_vec_env
from utils_my.sb3.my_evaluate_policy import evaluate_policy_with_success_rate

In [13]:
# Configuration
# Update these paths to point to your specific config and model
CONFIG_FILE_NAME = "configs/train/iteration_1/annealing/seed_1.json"
MODEL_PATH = "checkpoints/rl/iter1/simba_annealing/seed1/best_model"

EVAL_EPISODES = 50
SEED = 0
DETERMINISTIC = True

config_path = PROJECT_ROOT_DIR / CONFIG_FILE_NAME
model_path = PROJECT_ROOT_DIR / MODEL_PATH

print(f"Config: {config_path}")
print(f"Model: {model_path}")

Config: /home/hs/dev_ai/codes/IRPO/exp_on_flycraft/configs/train/iteration_1/annealing/seed_1.json
Model: /home/hs/dev_ai/codes/IRPO/exp_on_flycraft/checkpoints/rl/iter1/simba_annealing/seed1/best_model


In [14]:
# Initialize Environment
from flycraft.utils_common.load_config import load_config

# Load training config to get the environment config path
train_config = load_config(config_path)
env_config_file_name = train_config["env"]["config_file"]
env_config_path = PROJECT_ROOT_DIR / "configs" / "env" / env_config_file_name

print(f"Loading env config from: {env_config_path}")

# Using get_vec_env to ensure same wrappers (ScaledObservationWrapper, ScaledActionWrapper) as training
env = get_vec_env(
    num_process=1, 
    seed=SEED, 
    config_file=env_config_path,
    custom_config={"debug_mode": False}
)

print("Environment initialized.")

Loading env config from: /home/hs/dev_ai/codes/IRPO/exp_on_flycraft/configs/env/env_config_for_ppo.json
load config from: /home/hs/dev_ai/codes/IRPO/exp_on_flycraft/configs/env/env_config_for_ppo.json
0 Generator(PCG64) Generator(PCG64)
Environment initialized.


In [15]:
# Load Model
model = PPOWithBCLoss.load(
    model_path, 
    env=env,
    policy=SimBaActorCriticPolicy,
    custom_objects={
        "observation_space": env.observation_space,
        "action_space": env.action_space
    }
)

print("Model loaded successfully.")

verbose:  0
Model loaded successfully.


In [16]:
# Run Evaluation
print(f"Evaluating for {EVAL_EPISODES} episodes...")

mean_reward, std_reward, success_rate = evaluate_policy_with_success_rate(
    model=model,
    env=env,
    n_eval_episodes=EVAL_EPISODES,
    deterministic=DETERMINISTIC
)

print("-" * 50)
print(f"Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"Success Rate: {success_rate * 100:.2f}%")
print("-" * 50)

Evaluating for 50 episodes...


/home/hs/dev_ai/codes/IRPO/exp_on_flycraft/utils_my/sb3/my_evaluate_policy.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


--------------------------------------------------
Mean Reward: -225.44 +/- 53.38
Success Rate: 2.00%
--------------------------------------------------
